# Step 7: Infer gender from the PKG forenames (Wiki-Gendersort)

Every author × paper row extracted in step 6 is classified by Wiki-Gendersort on its PKG forename (`ForeName`; HTML
entities decoded, surrounding blanks removed). Each distinct forename is classified only once (identical result to one
call per row). Forenames that consist of initials only are returned as `INI`, unknown names as `UNK`.

Gender per author (AID) = **majority over all papers** of the author: F if more F than M votes, M if more M than F votes,
`UNI` (conflict) if both are equal and > 0; without any F/M vote the most frequent of `UNI`/`INI`/`UNK`.
Authors classified as F, M or UNI here are final. All rows of the remaining authors (`INI`/`UNK`, or no forename at all)
go into the forename recovery from MEDLINE (step 8).

**Input:** `data/derived/author_names_pkg.tsv` (step 6)
**Outputs:**
`data/derived/gender_pkg_by_aid.csv` (columns `AID, n_f, n_m, gender_pkg`) – used in step 10;
`data/derived/authors_unclassified_pkg.tsv` (columns `AID, LastName, ForeName, gender, AuthorNum, AuthorOrder, PMID`;
`gender` = PKG code of the author) – input of step 8.

Requires Wiki-Gendersort: `git clone https://github.com/nicolasberube/Wiki-Gendersort.git` (a separate Python environment is recommended).

## 0. Configuration

In [ ]:
import os
import html
import duckdb
import pandas as pd

WG_DIR = 'path/to/Wiki-Gendersort'                # local clone of Wiki-Gendersort (adapt)
BASE_DIR = os.path.abspath('..')                  # root of the replication package (notebooks run from code/)
DERIVED_DIR = f'{BASE_DIR}/data/derived'
IN_TSV = f'{DERIVED_DIR}/author_names_pkg.tsv'                   # step 6
OUT_AID_CSV = f'{DERIVED_DIR}/gender_pkg_by_aid.csv'
OUT_RECOVERY_TSV = f'{DERIVED_DIR}/authors_unclassified_pkg.tsv'
DB_PATH = f'{DERIVED_DIR}/infer_gender_pkg.duckdb'

con = duckdb.connect(DB_PATH)
con.execute("SET preserve_insertion_order = false")

os.chdir(WG_DIR)                      # Wiki-Gendersort reads NamesOut.txt from its own folder
from Wiki_Gendersort import wiki_gendersort
WG = wiki_gendersort()

## 1. Load the PKG names

In [ ]:
con.execute(f"""
CREATE OR REPLACE TABLE names AS
SELECT CAST(AID AS BIGINT) AS aid, LastName, ForeName, AuthorNum, AuthorOrder, PMID
FROM read_csv('{IN_TSV}', delim = '\t', header = true, all_varchar = true)
""")
print(con.execute("SELECT count(*) AS n_rows, count(DISTINCT aid) AS n_authors, count(DISTINCT ForeName) AS n_forenames FROM names").df())

## 2. Classify every distinct forename once

In [ ]:
forenames = con.execute(
    "SELECT DISTINCT ForeName FROM names WHERE ForeName IS NOT NULL AND trim(ForeName) <> ''").df()

cache = {}
def classify(name):
    clean = html.unescape(name).strip()          # e.g. '&Zbreve;' -> 'Ž', leading blanks
    if clean not in cache:
        cache[clean] = WG.assign(clean)
    return cache[clean]

forenames['gender'] = [classify(n) for n in forenames['ForeName']]
con.register('forenames_df', forenames)
con.execute("CREATE OR REPLACE TABLE forename_gender AS SELECT ForeName, gender FROM forenames_df")
print(forenames['gender'].value_counts(dropna=False))

## 3. Gender per author (majority vote over all papers)

Rows without forename count as `UNK`.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE gender_pkg AS
WITH author_rows AS (
    SELECT n.aid, coalesce(f.gender, 'UNK') AS gender
    FROM names n
    LEFT JOIN forename_gender f USING (ForeName)
),
votes AS (
    SELECT aid,
           count(*) FILTER (WHERE gender = 'F') AS n_f,
           count(*) FILTER (WHERE gender = 'M') AS n_m,
           mode(gender) FILTER (WHERE gender IN ('UNI', 'INI', 'UNK')) AS other_mode
    FROM author_rows
    GROUP BY 1
)
SELECT aid, n_f, n_m,
       CASE WHEN n_f > n_m THEN 'F'
            WHEN n_m > n_f THEN 'M'
            WHEN n_f = n_m AND n_f > 0 THEN 'UNI'      -- conflicting votes
            ELSE other_mode END AS gender_pkg
FROM votes
""")
print(con.execute("""
SELECT gender_pkg, count(*) AS n_authors,
       round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct,
       avg(((n_f > 0) AND (n_m > 0))::INT) AS share_with_conflicting_votes
FROM gender_pkg GROUP BY 1 ORDER BY 2 DESC
""").df())

## 4. Export

In [ ]:
con.execute(f"""
COPY (SELECT aid AS AID, n_f, n_m, gender_pkg FROM gender_pkg ORDER BY aid)
TO '{OUT_AID_CSV}' (HEADER, DELIMITER ',')
""")

# all rows of authors without F/M/UNI -> forename recovery (step 8)
con.execute(f"""
COPY (
    SELECT n.aid AS AID, n.LastName, n.ForeName, g.gender_pkg AS gender, n.AuthorNum, n.AuthorOrder, n.PMID
    FROM names n
    JOIN gender_pkg g USING (aid)
    WHERE g.gender_pkg IS NULL OR g.gender_pkg NOT IN ('F', 'M', 'UNI')
) TO '{OUT_RECOVERY_TSV}' (HEADER, DELIMITER '\t')
""")
print(con.execute(f"SELECT count(*) AS n_rows, count(DISTINCT AID) AS n_authors FROM read_csv('{OUT_RECOVERY_TSV}', delim = '\\t', header = true)").df())
con.close()